# <span style="color:#2E86C1">CP5 · Fer la prova final i explicar el resultat</span>

**Objectiu:** comprovar el model en dades reservades i comunicar els seus errors i límits.

## <span style="color:#16A085">1. Deixar per escrit el model, els predictors i els paràmetres abans d’executar la prova.</span>

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- DECISIÓ CONGELADA (vinguda de CP3, apartat 5.4) ----------
# Model:        Ridge (regressió lineal regularitzada, amb NumPy)
# Paràmetre:    alpha = 300 (triat només amb la validació de 2025)
# Entrenament:  2022-2024  (2021 només serveix per a crear lag_12)
# Validació:    2025       (ja utilitzada per a triar; no es toca més)
# Prova final:  gener-juny 2026 (s'avalua UNA sola vegada)
# Predictors (es coneixen abans del mes):
#   numèrics:    marketing_budget, planned_discount, stock_units, list_price,
#                days_in_month, lag_1, lag_12
#   categòrics:  channel, product_line, region, campaign
#   derivats:    month_sin, month_cos, time_index i sinus/cosinus per producte
# Excloses (se saben després): revenue, website_sessions, marketing_spend,
#   avg_discount, closing_stock_units, stockout
# Les prediccions es limiten entre 0 i l'estoc inicial.
ALPHA = 300
KEY = ['channel', 'product_line', 'region']
NUM = ['marketing_budget', 'planned_discount', 'stock_units', 'list_price',
       'days_in_month', 'lag_1', 'lag_12']
CAT = KEY + ['campaign']

# ---------- Dades i particions ----------
sales = pd.read_csv(Path('..') / 'dades' / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)

for lag in [1, 12]:
    sales[f'lag_{lag}'] = sales.groupby(KEY)['sales_units'].shift(lag)
usable = sales.dropna(subset=['lag_1', 'lag_12'])

train = usable[usable['month'] < '2025-01-01'].copy()
test = usable[usable['month'] >= '2026-01-01'].copy()
print('Entrenament:', len(train), 'files · Prova:', len(test), 'files')
print('Mesos de la prova:', test['month'].dt.strftime('%Y-%m').unique())

# ---------- Funcions (les mateixes que en CP3) ----------
def score(y, pred):
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    return {'MAE': np.mean(np.abs(y - pred)), 'WAPE_%': 100 * np.abs(y - pred).sum() / y.sum()}

def cap(frame, pred):
    return np.clip(pred, 0, frame['stock_units'].to_numpy())

def design(frame):
    x = frame[NUM].copy()
    month = frame['month'].dt.month
    x['month_sin'] = np.sin(2 * np.pi * month / 12)
    x['month_cos'] = np.cos(2 * np.pi * month / 12)
    x['time_index'] = (frame['month'].dt.year - 2021) * 12 + month - 1
    for p in ['urban', 'mountain', 'electric']:
        x[f'{p}_sin'] = x['month_sin'] * (frame['product_line'] == p)
        x[f'{p}_cos'] = x['month_cos'] * (frame['product_line'] == p)
    return pd.concat([x, pd.get_dummies(frame[CAT], dtype=float)], axis=1)

def fit_ridge(frame, alpha):
    x = design(frame)
    medians = x.median()
    x = x.fillna(medians)
    center, scale = x.mean(), x.std().replace(0, 1)
    z = np.column_stack([np.ones(len(x)), ((x - center) / scale).to_numpy(dtype=float)])
    penalty = np.eye(z.shape[1]) * alpha
    penalty[0, 0] = 0
    beta = np.linalg.solve(z.T @ z + penalty, z.T @ frame['sales_units'].to_numpy())
    return x.columns, medians, center, scale, beta

def predict_ridge(frame, model):
    columns, medians, center, scale, beta = model
    x = design(frame).reindex(columns=columns, fill_value=0).fillna(medians)
    z = np.column_stack([np.ones(len(x)), ((x - center) / scale).to_numpy(dtype=float)])
    return z @ beta



Entrenament: 648 files · Prova: 108 files
Mesos de la prova: <StringArray>
['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
Length: 6, dtype: str


## <span style="color:#16A085">2. Avaluació gener–juny de 2026 d'una vegada amb el model creat amb anterioritat</span>

In [3]:
# Un sol ajust, només amb entrenament (2022-2024)
model = fit_ridge(train, ALPHA)

# Una sola avaluació de la prova (gener-juny 2026)
pred = cap(test, predict_ridge(test, model))
final = score(test['sales_units'], pred)
print(f"Prova final · MAE: {final['MAE']:.2f} unitats · WAPE: {final['WAPE_%']:.1f} %")

# Només per a comparar (no per a triar): el mateix mes de l'any anterior
ref = score(test['sales_units'], cap(test, test['lag_12']))
print(f"Referència lag_12 · MAE: {ref['MAE']:.2f} unitats · WAPE: {ref['WAPE_%']:.1f} %")
print("Validació 2025 (CP3) · MAE: 11.92")

Prova final · MAE: 16.28 unitats · WAPE: 13.7 %
Referència lag_12 · MAE: 21.24 unitats · WAPE: 17.8 %
Validació 2025 (CP3) · MAE: 11.92


## <span style="color:#16A085">3. Comparació del resultat amb validació i anàlisis de dos casos d’error.</span>

### <span style="color:#D68910">3.1 Comparació del resultat amb validació</span>

Per a comparar el resultat amb la validació procedim a entrenar el model de Ridge amb la funció creada anteriorment que entrena el model ridge

In [4]:
model = fit_ridge(train, ALPHA)

Seguidament usarem les funcions de `score` creades abans per a mesurar el `MAE` i el `WAPE`

In [5]:
val_preds = cap(val, predict_ridge(val, model))
test_preds = cap(test, predict_ridge(test, model))

val_score = score(val['sales_units'], val_preds)
test_score = score(test['sales_units'], test_preds)

Per últim mostrarem el resultat entre l'any 2025 (Validació) amb el 2026 (Prova)

In [6]:
print(f"\nValidació (2025) -> MAE: {val_score['MAE']:.2f} | WAPE: {val_score['WAPE_%']:.2f}%")
print(f"Prova (2026)     -> MAE: {test_score['MAE']:.2f} | WAPE: {test_score['WAPE_%']:.2f}%")


Validació (2025) -> MAE: 11.92 | WAPE: 11.49%
Prova (2026)     -> MAE: 16.28 | WAPE: 13.65%


### <span style="color:#D68910">3.2 Anàlisis de dos casos d’error</span>

Primerament copiem el model de test per a seguidament agafar les prediccions fetes anteriorment i compararles amb les dades reals per a vore el marge d'error i després trenasformar en dades absolutes (sols dades positives `> 0`)

In [7]:
test_res = test.copy()
test_res['pred'] = test_preds
test_res['error'] = test_res['sales_units'] - test_res['pred']
test_res['abs_error'] = test_res['error'].abs()

Seguidament mirem les columnes que ens poden interesar i d'allí sabrem els 2 errors més critics en el model de test en l'any 2026

In [8]:
print(test_res.sort_values('abs_error', ascending=False)[cols].head(2).to_string(index=False))

     month   channel product_line region campaign  stock_units  sales_units       pred      error
2026-04-01 ecommerce        urban  metro   spring          356          328 219.314142 108.685858
2026-03-01 ecommerce        urban  metro     none          310          230 179.806778  50.193222


## <span style="color:#16A085">4. Factors que influeixen en el model i límits de la interpretació</span>

En este apartat és donara a coneixer els factors que influeixen en el model i els seus límits d'interpretació, no sols del model creat per a este projecte, sino que serveix de forma general a tots els models (però és mostraran exemples en el model del projecte com a exemple).

### <span style="color:#D68910">4.1 Factors que influeixen en el model</span>

Els factors que més influeixen en el model que anem a explicar inclouen els següents apartats:

- Qualitat i representativitat de les dades.
- Supòsits i simplificacions.
- Complexitat i nombre de paràmetres.

#### <span style="color:#D68910">4.1.1 Qualitat i representativitat de les dades</span>

Per a qualsevol model de IA les dades no soles son claus per al seu entrenament, sino que també son vitals per a poder fer prediccions de la millor forma possible, aleshores si les dades proporcionades a eixe model són:

- `Incorrectes`
- `Desactualitzades`
- `Incompletes`

Aleshores el model no resultara útil degut a que s'equivocara en gran mesura (`+ MAE`) i per consequència tindra una menor probabilitat d'encertar (`- R2`).

#### <span style="color:#D68910">4.1.2 Supòsits i simplificacions</span>

Cada model parteix de hipòtesis previes sobre el comportament de les dades (variables), per exemple suposar relacions lineals, alehores si els supòsits no coincideixen amb la realitat aleshores el model serà menys precis i tindra major probabilitat de errar les prediccions.

#### <span style="color:#D68910">4.1.3 Complexitat i nombre de paràmetres</span>

En complexitat ens referim a la capacitat del model amb certa capacitat d'entrenament, per exemple un model més simple és més probable de patir de subajustament (`underfitting`), això significa que no té la capacitat (almenys no total) de trobar patrons importants per a poder predeïr de forma adequada, mentre que un massa complex pot sufrir de sobreajustament (`overfitting`), això significa que va a memoritzar informació que no aprofitara al model en lloc d'aprendre la tendència general de les dades.

### <span style="color:#D68910">4.2 Límits d'interpretació</span>

Encara que un model tinga una capacitat de predicció matemàtica extremadament potent té fronteres molt clares pel que fa a com s'han d'interpretar les seues propies conclusions, que poden ser:

- Correlació no implica causalitat.
- Omissió de factors qualitatius o contextuals.
- Límits en sistemes caòtics o estocàstics.

#### <span style="color:#D68910">4.2.1 Correlació no implica causalitat</span>

Significa que encara que el model detecte una associació estadística forta entre dos variables això no significa que una siga estrictament la causa de l'altra, a vegades n'hi han variables confoses o latents les cuals no havien sigut contemplades.

#### <span style="color:#D68910">4.2.2 Omissió de factors qualitatius o contextuals</span>

Ningun model pot contindre totes les variables del mon real, això significa que és capaç de reaccionar a imprevistos d'última hora o altres elements ja siguen humans, cultural o polítics.

#### <span style="color:#D68910">4.2.3 Límits en sistemes caòtics o estocàstics</span>

En sistemes amb una gran tendencia al canvi com podria ser mercats financers la capacitat de predicció i interpretació d'aquest últim es degrada de forma accelerada quan més passa el temps.